# 🎙️ Gemma 4 12B Multimodal Punjabi ASR & Transliteration Pipeline (Colab Edition)
This notebook implements a unified speech-to-text and script transliteration pipeline using Google's **Gemma 4 12B-it** model, optimized for the **Google Colab** environment with Google Drive integration.

### Features:
1. **Google Drive Integration**: Mounts your Google Drive directly to access audio files and save transcripts.
2. **Interactive UI**: Dropdown menu to easily select from supported audio files in your project directory.
3. **Kaggle API Integration**: Securely retrieves your Kaggle API token from Colab Secrets, wipes any corrupted caches, and downloads model weights.
4. **4-Bit Quantization**: Loads the 12B model in 4-bit precision to fit within standard Colab T4 GPU VRAM limits.
5. **Memory-Optimization**: Uses sequential layer loading (`low_cpu_mem_usage=True`) to prevent Colab system RAM OOM crashes.
6. **Speech-to-Text (ASR)**: Uses Gemma 4's native, encoder-free multimodal architecture to transcribe raw Punjabi audio waveforms directly.
7. **Script Transliteration**: Converts the Gurmukhi script transcription into Devanagari script phonetically (keeping exact words/dialect intact).
8. **Anti-Hallucination Measures**: Implements strict formatting rules and greedy decoding (`do_sample=False`) to ensure high-fidelity outputs.

---
### 📁 Step 1: Mount Google Drive and Set Up Workspace
We mount Google Drive to access your Punjabi audio files and set the active working directory.

In [ ]:
# Mount Google Drive to securely access your Punjabi audio files
from google.colab import drive
import os

drive.mount('/content/drive')

# Define the path to your specific project folder in Google Drive
PROJECT_FOLDER = '/content/drive/MyDrive/Punjabi_ASR_Pipeline'

# Check if the folder exists, and build it if it doesn't
if not os.path.exists(PROJECT_FOLDER):
    os.makedirs(PROJECT_FOLDER)
    print(f"Successfully created a new project directory at: {PROJECT_FOLDER}")
else:
    print(f"Existing project directory successfully located at: {PROJECT_FOLDER}")

# Change the current working directory of the notebook to your project folder
os.chdir(PROJECT_FOLDER)
print(f"Current working directory is now set to: {os.getcwd()}")

### 🎵 Step 2: Dynamic Dropdown Menu for Audio File Selection
Scan the current working directory for matching audio files and choose the file to transcribe.

In [ ]:
import os
import ipywidgets as widgets
from IPython.display import display

# Define supported audio extensions for your pipeline
SUPPORTED_EXTENSIONS = ('.mp3', '.wav', '.m4a', '.flac', '.ogg')

# Scan the current working directory for matching audio files
audio_files = [f for f in os.listdir('.') if f.lower().endswith(SUPPORTED_EXTENSIONS)]

# Check if the folder contains any valid audio files
if not audio_files:
    print("⚠️ No supported audio files found in your current directory.")
    print("Please upload your Punjabi audio files to this folder in Google Drive and rerun this cell.")
else:
    # Create and display the interactive dropdown widget
    file_dropdown = widgets.Dropdown(
        options=audio_files,
        value=audio_files[0],
        description='Audio File:',
        disabled=False,
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='50%')
    )

    print("📁 Select the Punjabi audio file you want to transcribe:")
    display(file_dropdown)

### 🔑 Step 3: Kaggle API Authentication and Weight Retrieval
Retrieve the token from your Google Colab secrets key `Kaggle_api`, clean up any emoji characters, and download the gemma-4 model weights using Kaggle Hub.

In [ ]:
import os
import shutil
import kagglehub
from google.colab import userdata

# 1. Explicitly clear environment variables from background memory
for key in ["KAGGLE_USERNAME", "KAGGLE_KEY", "KAGGLE_API_TOKEN"]:
    if key in os.environ:
        del os.environ[key]

# 2. Hard-delete the local Kaggle configuration directory to remove the cached files
kaggle_config_dir = os.path.expanduser("~/.kaggle")
if os.path.exists(kaggle_config_dir):
    shutil.rmtree(kaggle_config_dir)
    print("🧹 Successfully wiped out the corrupted local Kaggle file cache.")

try:
    # 3. Retrieve the token from your 'Kaggle_api' secret
    raw_token = userdata.get('Kaggle_api')

    # 4. Programmatically isolate and strip out any accidental emojis or whitespaces
    token_value = "".join(character for character in raw_token if ord(character) < 128).strip()

    # 5. Reassign the completely clean token to the environment parameters
    os.environ["KAGGLE_API_TOKEN"] = token_value
    print("✅ Fresh, clean token successfully mapped to active runtime.")

    # 6. Trigger the clean model weight download loop
    print("⏳ Downloading Gemma 4 12B Unified weights (this can take a few minutes)...")
    model_path = kagglehub.model_download("google/gemma-4/transformers/gemma-4-12b-it")

    print(f"\n🎉 Model weights successfully cached at:")
    print(model_path)

except userdata.SecretNotFoundError:
    print("\n❌ Error: Could not detect your secret key.")
    print("Please verify that you checked the 'Notebook access' toggle switch next to your 'Kaggle_api' variable inside the Secrets (🔑) tab.")
except Exception as e:
    print(f"\n❌ Pipeline Connection Failure: {e}")
    print("Ensure you have accepted the Gemma 4 license terms directly on the Kaggle model page.")

### 🛠️ Step 4: Upgrade Quantization Engine and Model Execution Libraries
Install and upgrade `bitsandbytes`, `accelerate`, and `transformers` to support quantized loading and execution of the model.

In [ ]:
# Install/upgrade the quantization engine
!pip install -U bitsandbytes accelerate transformers

### 🧠 Step 5: Initialize Quantized Model and Processor
Load the model in 4-bit precision to fit within the memory limits of a standard T4 GPU runtime, using `low_cpu_mem_usage=True` to prevent system OOM crashes, and initialize the multimodal processor.

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig

# Configure strict 4-bit quantization parameters for the T4 GPU profile
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("⏳ Loading Gemma 4 12B multi-modal processor and layers into memory...")

# Load the matching token/waveform processors using the cached directory path
processor = AutoProcessor.from_pretrained(model_path)

# Initialize causal language layers mapped to the hardware interface
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True  # 🛠️ FIXED: Sequential layer loading avoids system RAM OOM crash
)

print("✅ Gemma 4 12B Unified model successfully loaded into memory!")

### 📊 Step 6: Load and Pre-process Target Audio Waveforms
Load the selected audio file using `librosa`, resample it down to 16,000 Hz mono as required by Gemma 4's audio interface, and normalize amplitude coefficients.

In [ ]:
import librosa
import numpy as np

# Retrieve the file selected in your dropdown UI element
target_audio_file = file_dropdown.value
print(f"🎵 Accessing file reference target: {target_audio_file}")

print("⏳ Transforming audio track variables to match pipeline definitions...")

# Transform: Resample the incoming file array down to a strict 16,000 Hz mono channel layout
audio_waveform, sample_rate = librosa.load(target_audio_file, sr=16000, mono=True)

# Normalize structural amplitude coefficients between the [-1.0, 1.0] float range boundaries
if np.max(np.abs(audio_waveform)) > 0:
    audio_waveform = audio_waveform / np.max(np.abs(audio_waveform))

# Calculate timeline metadata bounds
total_duration = len(audio_waveform) / sample_rate

print("✅ Data transformations complete.")
print(f"📊 Sample Rate Matrix: {sample_rate} Hz | Unified Total Duration: {total_duration:.2f} seconds")

### 🗣️ Step 7: Run Deterministic ASR & Script Transliteration
We run a two-stage evaluation:
1. **Automatic Speech Recognition (ASR)**: Transcribes the first 30 seconds of audio into Gurmukhi Punjabi.
2. **Script Transliteration**: Phonetically converts Gurmukhi script to Devanagari script.

We configure the model to use **greedy decoding (`do_sample=False`)** and separate system prompt blocks to fully eliminate hallucinations, preambles, and filler text.

In [ ]:
import torch

print("⏳ Initializing pipeline functional tests...")

# Isolate the leading 30-second window matrix sequence boundary to run our performance check
max_test_samples = 16000 * 30
audio_test_chunk = audio_waveform[:max_test_samples]

# --- Stage 1: Automatic Speech Recognition (ASR) Passage ---
# Setting up system instructions and user message to keep output structured and verbatim
asr_system_instruction = (
    "You are a high-precision Automatic Speech Recognition model. Transcribe the following speech segment in Punjabi into Punjabi text (Gurmukhi script).\n"
    "Strict Rules:\n"
    "1. Only output the exact transcription. Do NOT translate the meaning.\n"
    "2. Do NOT add any preamble, conversational commentary, or explanations.\n"
    "3. Output the raw text directly, with no newlines."
)

messages = [
    {"role": "system", "content": asr_system_instruction},
    {
        "role": "user",
        "content": [
            {"type": "audio", "audio": audio_test_chunk}
        ]
    }
]

# Use the template engine to automatically inject structural placeholder tags into the prompt sequence
templated_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Pass the structurally tagged text sequence alongside the raw audio features into the processor
inputs = processor(text=templated_prompt, audio=audio_test_chunk, sampling_rate=16000, return_tensors="pt").to("cuda")

# 📊 Diagnostic Profile: Print the keys and dtypes to verify tensor structures BEFORE casting
print("\n📊 Inspecting Input Tensor Profiles (Before Casting):")
for key, value in inputs.items():
    if isinstance(value, torch.Tensor):
        print(f"  • {key}: dtype={value.dtype}, shape={list(value.shape)}")

# 🛠️ Direct Casting for Multimodal Inputs
# Only input_features should be bfloat16 to match the model's compute dtype.
# input_features_mask must remain a boolean tensor for indexing operations.
if "input_features" in inputs and inputs["input_features"].dtype == torch.float32:
    print(f"Casting input_features from {inputs['input_features'].dtype} to {torch.bfloat16}")
    inputs["input_features"] = inputs["input_features"].to(torch.bfloat16)

# 📊 Diagnostic Profile: Print the keys and dtypes to verify tensor structures AFTER casting
print("\n📊 Inspecting Input Tensor Profiles (After Casting):")
for key, value in inputs.items():
    if isinstance(value, torch.Tensor):
        print(f"  • {key}: dtype={value.dtype}, shape={list(value.shape)}")

print("\n🗣️ Generating Punjabi Text Transcription (Stage 1)...")
with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False  # Crucial: Disable sampling to prevent hallucinations
    )

# Track the exact token prompt length to cleanly slice off and decode ONLY the newly generated text responses
input_len = inputs["input_ids"].shape[1]
gurmukhi_output = processor.decode(generated_ids[0][input_len:], skip_special_tokens=True).strip()
print(f"\n✍️ Gurmukhi Script Output:\n{gurmukhi_output}")

# --- Stage 2: Structural Script Transliteration Passage ---
translit_system_instruction = (
    "You are a high-precision linguistic conversion model. Transliterate the provided Gurmukhi script (Punjabi) into Devanagari script (Hindi characters).\n\n"
    "Strict Rules:\n"
    "1. Do NOT translate the language into formal textbook Hindi. Keep the exact spoken words, colloquial expressions, and dialect intact—only change the script alphabets (e.g., 'ਵੀਰ ਜੀ, ਸਤਿ ਸ੍ਰੀ ਅਕਾਲ ਜੀ' must become 'वीर जी, सत श्री अकाल जी।').\n"
    "2. Preserve all punctuation, layout, and structural formatting exactly as they appear.\n"
    "3. Output ONLY the transliterated Devanagari text. Do not include any explanations, preambles, intros, or outros."
)

text_messages = [
    {"role": "system", "content": translit_system_instruction},
    {
        "role": "user",
        "content": [{"type": "text", "text": f"Gurmukhi Source Text:\n{gurmukhi_output}"}]
    }
]

templated_translit = processor.apply_chat_template(text_messages, tokenize=False, add_generation_prompt=True)
text_inputs = processor(text=templated_translit, return_tensors="pt").to("cuda")

print("\n🔀 Running Phonetic Script Transliteration into Devanagari (Stage 2)...")
with torch.no_grad():
    translit_ids = model.generate(
        **text_inputs,
        max_new_tokens=256,
        do_sample=False  # Crucial: Disable sampling to prevent hallucinations
    )

text_input_len = text_inputs["input_ids"].shape[1]
devanagari_output = processor.decode(translit_ids[0][text_input_len:], skip_special_tokens=True).strip()
print(f"\n🕉️ Final Devanagari Script Conversion Result:\n{devanagari_output}")